# 3. Desempenho e Complexidade de Algoritmos
Neste notebook, implementamos algoritmos clássicos de grafos e confrontamos a complexidade teórica com o tempo de execução empírico na rede real do *Memetracker*.

In [1]:
import networkx as nx
import pandas as pd
import time
import numpy as np
import scipy.stats as st
import random

# Carregando os dados e preparando a Maior Componente Conexa (LCC)
print("Carregando arestas...")
df = pd.read_csv('../data/edges_2days.csv', on_bad_lines='skip')
G_dir = nx.from_pandas_edgelist(df, source='source', target='target', create_using=nx.DiGraph())

G_undir = G_dir.to_undirected()
G_undir.remove_edges_from(nx.selfloop_edges(G_undir))

connected_components = list(nx.connected_components(G_undir))
lcc_nodes = max(connected_components, key=len)
LCC = G_undir.subgraph(lcc_nodes).copy()

print(f"Grafo Direcionado Original: {G_dir.number_of_nodes()} V, {G_dir.number_of_edges()} E")
print(f"Maior Componente Conexa (LCC): {LCC.number_of_nodes()} V, {LCC.number_of_edges()} E")

Carregando arestas...
Grafo Direcionado Original: 687347 V, 1193472 E
Maior Componente Conexa (LCC): 321645 V, 596233 E


### Ferramentas Estatísticas
Funções para realizar as rodadas experimentais e calcular o tempo médio, desvio padrão e Intervalos de Confiança (Normal ou t-Student).

In [2]:
def calc_stats(times):
    n = len(times)
    mean = np.mean(times)
    std = np.std(times, ddof=1) if n > 1 else 0.0
    
    alpha = 0.05
    if n >= 30:
        z = st.norm.ppf(1 - alpha/2)
        margin = z * (std / np.sqrt(n)) if n > 0 else 0
        dist_name = "Normal (Z)"
    else:
        t = st.t.ppf(1 - alpha/2, n-1) if n > 1 else 0
        margin = t * (std / np.sqrt(n)) if n > 0 else 0
        dist_name = "t-Student"
        
    return mean, std, margin, dist_name, n

def run_experiment(name, func, n_runs):
    print(f"Iniciando: {name} ({n_runs} rodadas)")
    times = []
    for i in range(n_runs):
        start = time.time()
        func(i)
        times.append(time.time() - start)
        
    mean, std, margin, dist_name, n = calc_stats(times)
    
    print(f"[Concluído] {name}")
    print(f"   Distribuição: {dist_name} (n={n})")
    print(f"   Média de Tempo: {mean:.5f}s")
    print(f"   Desvio Padrão:  {std:.5f}s")
    print(f"   IC (95%):       {mean:.5f}s +- {margin:.5f}s  [{mean-margin:.5f}s, {mean+margin:.5f}s]")
    print("-" * 60)

### Buscas em Grafos
Comparamos a Busca em Largura (BFS) e Busca em Profundidade (DFS) na Maior Componente Conexa.

In [3]:
nodes_list = list(LCC.nodes())
random.seed(42)
sources_30 = random.sample(nodes_list, 30)

run_experiment("Busca em Largura (BFS)", lambda i: list(nx.bfs_edges(LCC, sources_30[i])), 30)
run_experiment("Busca em Profundidade (DFS)", lambda i: list(nx.dfs_edges(LCC, sources_30[i])), 30)
run_experiment("Verificação de Eulerianidade", lambda i: nx.is_eulerian(G_undir), 30)

Iniciando: Busca em Largura (BFS) (30 rodadas)
[Concluído] Busca em Largura (BFS)
   Distribuição: Normal (Z) (n=30)
   Média de Tempo: 2.28150s
   Desvio Padrão:  0.43955s
   IC (95%):       2.28150s +- 0.15729s  [2.12421s, 2.43879s]
------------------------------------------------------------
Iniciando: Busca em Profundidade (DFS) (30 rodadas)
[Concluído] Busca em Profundidade (DFS)
   Distribuição: Normal (Z) (n=30)
   Média de Tempo: 0.90988s
   Desvio Padrão:  0.20675s
   IC (95%):       0.90988s +- 0.07398s  [0.83590s, 0.98386s]
------------------------------------------------------------
Iniciando: Verificação de Eulerianidade (30 rodadas)
[Concluído] Verificação de Eulerianidade
   Distribuição: Normal (Z) (n=30)
   Média de Tempo: 0.00003s
   Desvio Padrão:  0.00018s
   IC (95%):       0.00003s +- 0.00007s  [-0.00003s, 0.00010s]
------------------------------------------------------------


### Árvores Geradoras Mínimas (AGM) e Componentes Fortemente Conexas
Avaliamos o desempenho de Prim e Kruskal. Também rodamos o Algoritmo de Tarjan no grafo *direcionado* original.

In [4]:
run_experiment("AGM - Algoritmo de Prim", lambda i: nx.minimum_spanning_tree(LCC, algorithm='prim'), 30)
run_experiment("AGM - Algoritmo de Kruskal", lambda i: nx.minimum_spanning_tree(LCC, algorithm='kruskal'), 30)
run_experiment("Algoritmo de Tarjan (SCC)", lambda i: list(nx.strongly_connected_components(G_dir)), 30)

Iniciando: AGM - Algoritmo de Prim (30 rodadas)
[Concluído] AGM - Algoritmo de Prim
   Distribuição: Normal (Z) (n=30)
   Média de Tempo: 9.69578s
   Desvio Padrão:  1.86252s
   IC (95%):       9.69578s +- 0.66648s  [9.02930s, 10.36227s]
------------------------------------------------------------
Iniciando: AGM - Algoritmo de Kruskal (30 rodadas)
[Concluído] AGM - Algoritmo de Kruskal
   Distribuição: Normal (Z) (n=30)
   Média de Tempo: 9.15787s
   Desvio Padrão:  1.03931s
   IC (95%):       9.15787s +- 0.37190s  [8.78596s, 9.52977s]
------------------------------------------------------------
Iniciando: Algoritmo de Tarjan (SCC) (30 rodadas)
[Concluído] Algoritmo de Tarjan (SCC)
   Distribuição: Normal (Z) (n=30)
   Média de Tempo: 6.67130s
   Desvio Padrão:  0.97089s
   IC (95%):       6.67130s +- 0.34742s  [6.32388s, 7.01872s]
------------------------------------------------------------


### Caminhos Mínimos em Subgrafos
Devido à complexidade cúbica do Floyd-Warshall e aos gargalos do Dijkstra, extraímos subgrafos localizados para poder comparar o tempo de execução de Dijkstra e Bellman-Ford, e uma micro-rede para o Floyd-Warshall.

In [5]:
max_node = max(dict(LCC.degree()).items(), key=lambda x: x[1])[0]
sub_5k_nodes = list(nx.bfs_tree(LCC, max_node, depth_limit=3).nodes())[:5000]
sub_5k = LCC.subgraph(sub_5k_nodes).copy()
print(f"Subgrafo 5K: {sub_5k.number_of_nodes()} V, {sub_5k.number_of_edges()} E")
n_runs_5k = min(30, sub_5k.number_of_nodes())
sources_sub_30 = random.sample(list(sub_5k.nodes()), n_runs_5k)

sub_200_nodes = list(nx.bfs_tree(LCC, max_node, depth_limit=2).nodes())[:200]
sub_200 = LCC.subgraph(sub_200_nodes).copy()
print(f"Subgrafo 200: {sub_200.number_of_nodes()} V, {sub_200.number_of_edges()} E\n")

run_experiment("Caminhos Mínimos: BFS (Baseline 5K)", lambda i: list(nx.bfs_edges(sub_5k, sources_sub_30[i])), n_runs_5k)
run_experiment("Caminhos Mínimos: Dijkstra (5K)", lambda i: nx.single_source_dijkstra_path_length(sub_5k, sources_sub_30[i], weight=None), n_runs_5k)
run_experiment("Caminhos Mínimos: Bellman-Ford (5K)", lambda i: nx.single_source_bellman_ford_path_length(sub_5k, sources_sub_30[i], weight=None), n_runs_5k)

run_experiment("Floyd-Warshall (APSP 200)", lambda i: nx.floyd_warshall(sub_200, weight=None), min(10, sub_200.number_of_nodes()))

Subgrafo 5K: 5000 V, 13487 E
Subgrafo 200: 200 V, 212 E

Iniciando: Caminhos Mínimos: BFS (Baseline 5K) (30 rodadas)
[Concluído] Caminhos Mínimos: BFS (Baseline 5K)
   Distribuição: Normal (Z) (n=30)
   Média de Tempo: 0.01123s
   Desvio Padrão:  0.00214s
   IC (95%):       0.01123s +- 0.00077s  [0.01046s, 0.01200s]
------------------------------------------------------------
Iniciando: Caminhos Mínimos: Dijkstra (5K) (30 rodadas)
[Concluído] Caminhos Mínimos: Dijkstra (5K)
   Distribuição: Normal (Z) (n=30)
   Média de Tempo: 0.02578s
   Desvio Padrão:  0.00171s
   IC (95%):       0.02578s +- 0.00061s  [0.02516s, 0.02639s]
------------------------------------------------------------
Iniciando: Caminhos Mínimos: Bellman-Ford (5K) (30 rodadas)
[Concluído] Caminhos Mínimos: Bellman-Ford (5K)
   Distribuição: Normal (Z) (n=30)
   Média de Tempo: 0.03950s
   Desvio Padrão:  0.00394s
   IC (95%):       0.03950s +- 0.00141s  [0.03809s, 0.04091s]
----------------------------------------------